<a href="https://colab.research.google.com/github/HashamHassan-01/flyrank-ml-internship-hasham/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HashamHassan-01/flyrank-ml-internship-hasham/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1

The paper reports that growing content tends to be longer, younger, and already positioned slightly better in search than declining content. The authors also state that this is an observational comparison based on a large portfolio rather than proof of causation.

### My Methodology Question

Where does the "growing" versus "declining" label come from? Is it based only on changes in impressions over a fixed time window, and were other factors such as seasonal trends or algorithm updates considered? Understanding how these labels were created would help determine how well the comparison supports the conclusion.

### Finding 2

The paper reports that content performs best around 61–90 days after publication, declines after about 270 days, and that older content can recover when it is refreshed.

### My Methodology Question

How was the validation performed to distinguish the effect of refreshing content from other factors? For example, were refreshed pages compared with similar pages that were not refreshed during the same period? This would strengthen the evidence that the observed improvement is associated with the refresh rather than unrelated changes.

In [ ]:
import os

if not os.path.exists("flyrank-ml-internship-hasham"):
    !git clone https://github.com/HashamHassan-01/flyrank-ml-internship-hasham.git

%cd flyrank-ml-internship-hasham

Cloning into 'flyrank-ml-internship-hasham'...
remote: Enumerating objects: 178, done.
remote: Counting objects: 100% (178/178), done.
remote: Compressing objects: 100% (135/135), done.
remote: Total 178 (delta 78), reused 92 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (178/178), 1.89 MiB | 12.34 MiB/s, done.
Resolving deltas: 100% (78/78), done.
/content/flyrank-ml-internship-hasham/flyrank-ml-internship-hasham/flyrank-ml-internship-hasham/flyrank-ml-internship-hasham/flyrank-ml-internship-hasham


In [ ]:
df = pd.read_csv("./data/raw/content_refresh_anonymized.csv")

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

## Honest Validation

In Week 5, I evaluated my Random Forest Regressor using a random 80/20 train-test split.

For this validation audit, I re-ran the model using a GroupShuffleSplit based on `client_id`. This prevents pages from the same client appearing in both the training and testing sets, giving a more realistic estimate of how the model may perform on unseen clients.

The results below compare the original random split with the grouped validation approach.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
df = pd.read_csv("./data/raw/content_refresh_anonymized.csv")

# Create normalized scores
df["volume_score"] = df["search_volume"] / df["search_volume"].max()
df["age_score"] = df["content_age_days"] / df["content_age_days"].max()
df["rank_score"] = df["avg_position"] / df["avg_position"].max()
df["ctr_score"] = 1 - (df["ctr"] / df["ctr"].max())

# Create target
df["baseline_score"] = (
    0.40 * df["volume_score"] +
    0.25 * df["rank_score"] +
    0.20 * df["age_score"] +
    0.15 * df["ctr_score"]
)

# Remove missing values
df = df.dropna(subset=[
    "search_volume",
    "content_age_days",
    "avg_position",
    "ctr",
    "baseline_score"
])

# Features and target
X = df[[
    "search_volume",
    "content_age_days",
    "avg_position",
    "ctr"
]]

y = df["baseline_score"]

In [ ]:
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=df["client_id"])
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Grouped Validation Results")
print(f"MAE : {mae:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"R²  : {r2:.6f}")

Grouped Validation Results
MAE : 0.002135
RMSE: 0.006438
R²  : 0.987548


In [ ]:
print("Random Split (Week 5 / ML-08 baseline)")
print(f"MAE : {0.000626:.6f}")
print(f"RMSE: {0.003581:.6f}")
print(f"R²  : {0.995588:.6f}")

print("\nGrouped Split (this week, by client_id)")
print(f"MAE : {mae:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"R²  : {r2:.6f}")

Random Split (Week 5 / ML-08 baseline)
MAE : 0.000626
RMSE: 0.003581
R²  : 0.995588

Grouped Split (this week, by client_id)
MAE : 0.002135
RMSE: 0.006438
R²  : 0.987548


Before/After Comparison
Under the random 80/20 split (Week 5), the model achieved MAE 0.000626, RMSE 0.003581, R² 0.995588. Under the grouped split by client_id (this week), MAE rose to 0.002135 and RMSE to 0.006438, with R² dropping slightly to 0.987548. Error more than tripled under the honest split, consistent with the random split partly benefiting from client-specific patterns leaking across train and test.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*


I reviewed the features used by my final model to check for possible data leakage.

The model uses four input features: `search_volume`, `content_age_days`, `avg_position`, and `ctr`. These variables are available before calculating the Opportunity Score and do not directly contain the target value.

However, the target (`baseline_score`) was created from these same four features using a weighted formula. This means the model is learning to reproduce a deterministic score rather than predicting an independent real-world outcome. While this is not direct leakage from future information, it does make the prediction task much easier and explains the very high evaluation scores.

Identifier columns such as `content_id` and `client_id` were not used as input features, reducing the risk of the model memorizing specific pages or clients.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Features used by the model:")
print(X.columns.tolist())

print("\nTarget column:")
print("baseline_score")


Features used by the model:
['search_volume', 'content_age_days', 'avg_position', 'ctr']

Target column:
baseline_score


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*


### Original Claim

The Random Forest model accurately predicts the best pages to refresh.

### Revised Claim

The model achieved high predictive performance on the constructed baseline score under both random and grouped validation. Because the target score was derived from the same input features, these results should be interpreted as demonstrating the model's ability to reproduce the scoring formula rather than proving real-world SEO effectiveness.

The observed results suggest that the model may serve as a decision-support tool for prioritizing content refresh opportunities, but additional validation using independent outcome data would be required before deployment.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
pass



## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it.
- [x] The notebook runs top to bottom with no errors (Runtime → Run all).
- [x] No client names, URLs, or private queries anywhere.
- [x] My claims use careful words: observed, measured, directional, decision-support.
- [x] Committed to my repo under `work/notebooks/` — then submit my repo URL on the card. Done.